# Paper R5: SHAP on each split's 500-row test core (native and common settings)

Conference paper, Phase 5 (decisions D33, D35, D36, D37 and its amendment; claude/paper/phase4_design.md sections 3, 9, 11, 12). **Runs on the PC only.** It explains models; it fits nothing new except the ALT_XGB baseline (R4 kept no baseline models) and, if a saved R4 pipeline is missing, a refit of the selected configuration.

What it does, in this order:
1. **Gates:** data, Stage 4 freeze and pipelines, `test_predictions.csv`, Stage 1 index files, every R4 file against `SHA256SUMS_r4.txt`, the split 2–5 index files against `MANIFEST_splits2_3_4_5.csv`, and the Stage 6 SHAP files against their `.done.json` hashes.
2. **Models:** split 1 = the frozen thesis pipelines; splits 2–5 = refits of the selected configuration on the split's development rows (R4 also saved these six pipelines on the PC, `paper_r4_outputs/pipelines/`, hash in `summary.json`; their hash is recorded for information, they are not explained). ALT_XGB is refitted per split (R4 saved no baseline models). **Every model must reproduce its stored test predictions exactly (full test set) before any SHAP, or the run stops.**
3. **Regression check:** the R5 code on the first 20 split-1 core rows must reproduce the Stage 6 values (tree/linear within 1e-10; permutation exactly).
4. **Native setting** (as in Stage 6): TreeSHAP (interventional, the split's 500 background; log-odds for XGB/LGBM/CAT, probability for RF), LinearExplainer for LR, permutation for MLP (100 background, max_evals 250). Split 1 reuses the Stage 6 values (hash-checked). ALT_XGB native on every split.
5. **Common setting:** PermutationExplainer on `predict_proba[:, 1]` for all six models, the split's 100-row background, max_evals 250, one seed per split (D37 amendment B5) and one per row. Split 1 also gets the XGB check (permutation on log-odds vs exact TreeSHAP, same 100 background).
6. **Outputs:** per-row SHAP at 17 digits with base value, model output and additivity error; an additivity table; ALT_XGB top-5 sets. Agreement metrics are computed later in the workspace.

**Resumable:** every task is saved in row chunks under `paper_r5_outputs/_work/` with a `.done.json` written last. If the run stops, Run All again: finished tasks and chunks are skipped and the files are byte-identical to an uninterrupted run. Cell 9 prints the plan and the time remaining.

**Run:** put the notebook in `thesis_xai/` (next to `paper_r4_outputs/` with its `pipelines/` folder, and `r4_upload/SHA256SUMS_r4.txt`), set `R5_CONFIRMED = True` in Cell 2, keep the laptop on mains power with sleep disabled, and Run All. Then run `python pack_r5_outputs.py` and upload `r5_upload/`.

In [1]:
# Cell 1 - environment guard (S10): before numpy / scikit-learn are imported
import os, sys
TEST_MODE = os.environ.get("R5_TEST_MODE") == "1"          # Claude's workspace test only; never set on the PC
PROJECT_DIR = os.environ.get("R5_PROJECT_DIR", os.path.abspath("."))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import env_setup
env_setup.ensure_pinned(PROJECT_DIR, install=False)

All pinned packages present at the pinned versions.


{}

In [2]:
# Cell 2 - configuration
R5_CONFIRMED = True                       # set True to run (D37 amendment approved)
if TEST_MODE:
    R5_CONFIRMED = True
if not R5_CONFIRMED:
    raise SystemExit("Set R5_CONFIRMED = True in Cell 2, then Run All.")

import contextlib, hashlib, io, json, platform, time, warnings
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
import shap
import sklearn
from joblib import Parallel, delayed
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import thesis_common as tc

MODEL_SOURCE = "refit"                     # C2 (D37 amendment): refit every selected configuration on the PC; "r4_pipelines"
                                           # loads the R4 pipelines saved on the PC (hash = summary.json) instead
FREEZE_SHA256 = "1dd5bf06708d148399675d9fc657857df1c86f015cbd83c24e79e8754b48c0d8"   # S4-047
PRED_SHA256 = "b9813461cd03e4552b50c245c8750648d08b9c48c1960d249642982bc15114ed"     # S5-136
SUMS_R4_SHA256 = "dc79a5bb2476664230310c68ad9befcf6a6c8a3160891b75b28b81cae1c55b12"      # SHA256SUMS_r4.txt as returned (D37)
MANIFEST_R4_SHA256 = "d04472a61f148527087be432731f13e4344e4e7e40a3026ed3c52e3764b45ca1"  # MANIFEST_splits2_3_4_5.csv
STAGE1_SHA = {                              # Stage 1 index files (MANIFEST.csv; rebuilt byte-identical, P2-001)
    "dev_folds.csv": "c71320238e98f1d8fedf1ce72475f8572e27524805201480857374ccc3ab7977",
    "shap_background_500_dev.csv": "981fe21fcd3e157857be057b0a9e43a93115fe99e10aff4313cbddd26e60a21c",
    "shap_perm_background_100_dev.csv": "a5c697d56d71fa26074110fef9e18f16cdfd3fbdff6c11741249c1bd6f466e09",
    "shap_perm_core_500_test.csv": "59dae554426fe5adbbc9330505ccfaa099b9022d9e8e7678d97a877952599fe9"}
if TEST_MODE and os.environ.get("R5_TEST_SUMS_SHA256"):
    SUMS_R4_SHA256 = os.environ["R5_TEST_SUMS_SHA256"]      # fixture list (test only)
MODELS6 = ["LR", "XGB", "LGBM", "MLP", "RF", "CAT"]
FEATURES = list(tc.FEATURES)
TREE = ["XGB", "LGBM", "RF", "CAT"]
SCALE_NATIVE = {"LR": "log-odds", "XGB": "log-odds", "LGBM": "log-odds", "CAT": "log-odds", "RF": "probability",
                "MLP": "probability", "ALT_XGB": "log-odds"}
SPLITS = [int(s) for s in os.environ.get("R5_SPLITS", "1,2,3,4,5").split(",")]
CAT_THREADS = 12                            # D35 item 1
MAX_EVALS = 250                             # P3
TREE_CHUNK, PERM_CHUNK = 50, 10
TREE_WORKERS = 6 if not TEST_MODE else max(1, min(6, (os.cpu_count() or 2)))
REGRESSION_ROWS = 20
MAX_ROWS = int(os.environ.get("R5_MAX_ROWS", "0")) if TEST_MODE else 0        # 0 = all 500 core rows
MAX_ROWS_MODEL = json.loads(os.environ.get("R5_MAX_ROWS_MODEL", "{}")) if TEST_MODE else {}
STOP_AFTER = os.environ.get("R5_STOP_AFTER", "") if TEST_MODE else ""       # test only: "task:chunk" interrupt
REFIT_TOL = float(os.environ.get("R5_REFIT_TOL", "0")) if TEST_MODE else 0.0
ALT_XGB_PARAMS = dict(n_estimators=250, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                      tree_method="hist", n_jobs=-1, random_state=tc.MODEL_SEED, verbosity=0)   # R4 Cell 8

# Seeds. S2 + 14th stream (shap_permutation, S15k) + 15th (paper_outer, D33); per split 7 words (R4) + an 8th word
# for the permutation explainer (D37 amendment B5). Split 1 uses the Stage 6 main permutation seed, so its MLP
# common-setting values equal the Stage 6 MLP values.
SS15 = np.random.SeedSequence(tc.MASTER_SEED).generate_state(15)
assert [int(v) for v in SS15[:13]] == [tc.SEEDS[k] for k in tc.SEED_STREAMS], "seed derivation changed"
assert int(SS15[13]) == 3957024168, "14th stream (shap_permutation) changed"
PAPER_OUTER = int(SS15[14])
SPLIT_STREAMS = ["partition", "folds", "subsample_50k", "shap_background", "shap_perm_background", "test_core", "eval_bootstrap"]
STAGE6_PERM_SEED = int(np.random.SeedSequence(int(SS15[13])).generate_state(2)[0])
assert STAGE6_PERM_SEED == 2941821818, "Stage 6 permutation seed changed"


def split_seeds(k):
    return {n: int(v) for n, v in zip(SPLIT_STREAMS, np.random.SeedSequence([PAPER_OUTER, k]).generate_state(7))}


def perm_seed(k):
    if k == 1:
        return STAGE6_PERM_SEED
    s8 = np.random.SeedSequence([PAPER_OUTER, k]).generate_state(8)
    assert [int(v) for v in s8[:7]] == list(split_seeds(k).values()), "the first 7 per-split words changed"
    return int(s8[7])


def row_seed(base, row_id):
    # one seed per explained row (S15k): results do not depend on chunking, order or interruptions
    return int(np.random.SeedSequence([int(base), int(row_id)]).generate_state(1)[0])


OUT = os.path.join(PROJECT_DIR, "paper_r5_TEST" if TEST_MODE else "paper_r5_outputs")
R4 = os.environ.get("R5_R4_DIR", os.path.join(PROJECT_DIR, "paper_r4_outputs"))
WORK, MODELS_DIR = os.path.join(OUT, "_work"), os.path.join(OUT, "models")
for d in [OUT, WORK, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
MACHINE = "workspace-test" if TEST_MODE else "pc"
if not TEST_MODE and (platform.system() != "Windows" or os.cpu_count() != 12):
    raise SystemExit("R5 runs on the project PC only (Windows, 12 logical CPUs): XGBoost and CatBoost models differ between machines.")
print(f"machine={MACHINE} python={platform.python_version()} shap={shap.__version__} tree workers={TREE_WORKERS} model source={MODEL_SOURCE}")
print("permutation seeds:", {k: perm_seed(k) for k in SPLITS})

C:\Users\USER\Downloads\thesis_xai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


machine=pc python=3.13.0 shap=0.52.0 tree workers=6 model source=refit
permutation seeds: {1: 2941821818, 2: 1798040441, 3: 1261013684, 4: 3055778493, 5: 2471599976}


In [3]:
# Cell 3 - gates (C1): data, freeze, Stage 5 predictions, Stage 1 index files, R4 files, Stage 6 SHAP files
def sha(path):
    return tc.sha256_file(path)


GATE = []
def gate(name, ok, detail=""):
    GATE.append({"check": name, "pass": bool(ok), "detail": str(detail)})
    if not ok:
        print("FAIL", name, detail)


X_all, y_all = tc.load_data(PROJECT_DIR)                                     # SHA-256 checked
FZ_PATH = os.path.join(PROJECT_DIR, "stage4_outputs", "frozen_models.json")
gate("Stage 4 freeze", sha(FZ_PATH) == FREEZE_SHA256)
FZ = json.load(open(FZ_PATH, encoding="utf-8"))
PIPE1 = {m: os.path.join(PROJECT_DIR, FZ["models"][m]["pipeline"]) for m in MODELS6}
for m in MODELS6:
    gate(f"split 1 frozen pipeline {m}", os.path.exists(PIPE1[m]) and sha(PIPE1[m]) == FZ["models"][m]["pipeline_sha256"])
PRED_PATH = os.path.join(PROJECT_DIR, "stage5_outputs", "test_predictions.csv")
gate("Stage 5 test_predictions.csv", os.path.exists(PRED_PATH) and sha(PRED_PATH) == PRED_SHA256)
PRED = pd.read_csv(PRED_PATH, float_precision="round_trip")
IDX1 = os.path.join(PROJECT_DIR, "stage1_outputs", "indices")
for f, h in STAGE1_SHA.items():
    gate(f"Stage 1 {f}", os.path.exists(os.path.join(IDX1, f)) and sha(os.path.join(IDX1, f)) == h)

# every R4 file listed in SHA256SUMS_r4.txt (entries "zip:path"; the files are checked in paper_r4_outputs/)
SUMS_PATH = next((p for p in [os.path.join(PROJECT_DIR, "r4_upload", "SHA256SUMS_r4.txt"), os.path.join(R4, "SHA256SUMS_r4.txt"),
                              os.path.join(PROJECT_DIR, "SHA256SUMS_r4.txt")] if os.path.exists(p)), None)
gate("SHA256SUMS_r4.txt found and unchanged", SUMS_PATH is not None and sha(SUMS_PATH) == SUMS_R4_SHA256, SUMS_PATH)
n_files = n_ok = n_zip = n_zip_ok = 0
nb_note = "not checked"
for line in open(SUMS_PATH, encoding="utf-8").read().splitlines():
    h, name = line.split("  ", 1)
    if ":" not in name:                                                        # a zip: checked if it is still on the PC
        zp = os.path.join(os.path.dirname(SUMS_PATH), name)
        if os.path.exists(zp):
            n_zip += 1; n_zip_ok += sha(zp) == h
        continue
    zname, rel = name.split(":", 1)
    if rel == "paper_r4_splits.ipynb":                                         # executed notebook: informative only
        p = os.path.join(PROJECT_DIR, rel)
        nb_note = "identical" if os.path.exists(p) and sha(p) == h else "changed or absent (not a stop)"
        continue
    n_files += 1
    p = os.path.join(R4, rel)
    ok = os.path.exists(p) and sha(p) == h
    n_ok += ok
    if not ok:
        print("  differs or missing:", rel)
gate("R4 files vs SHA256SUMS_r4.txt", n_ok == n_files and n_files > 0, f"{n_ok}/{n_files}")
gate("R4 zips still on the PC vs SHA256SUMS_r4.txt", n_zip_ok == n_zip, f"{n_zip_ok}/{n_zip} present")
MAN_PATH = os.path.join(R4, "indices", "MANIFEST_splits2_3_4_5.csv")
gate("MANIFEST_splits2_3_4_5.csv", os.path.exists(MAN_PATH) and sha(MAN_PATH) == MANIFEST_R4_SHA256)
MAN4 = pd.read_csv(MAN_PATH)
n_idx = sum(sha(os.path.join(R4, "indices", f"split{r.split}", r.file)) == r.sha256 for r in MAN4.itertuples())
gate("split 2-5 index files vs MANIFEST", n_idx == len(MAN4), f"{n_idx}/{len(MAN4)}")

# Stage 6 SHAP files (split-1 native values are reused): each file against its own .done.json record
S6 = os.path.join(PROJECT_DIR, "stage6_outputs")
S6_FILES = {}
for m in MODELS6:
    dp = os.path.join(S6, "shap", f"SHAP_{m}.done.json")
    ok = os.path.exists(dp)
    if ok:
        rec = json.load(open(dp, encoding="utf-8"))
        S6_FILES[m] = os.path.join(S6, rec["file"])
        ok = os.path.exists(S6_FILES[m]) and sha(S6_FILES[m]) == rec["sha256"]
    gate(f"Stage 6 SHAP_{m}.csv vs its .done.json", ok)
GATE_DF = pd.DataFrame(GATE)
print(GATE_DF.to_string(index=False))
print("executed R4 notebook:", nb_note)
if not GATE_DF["pass"].all():
    raise SystemExit("Input gates failed; nothing was explained. Report the FAIL lines.")

                                       check  pass                                                                detail
                              Stage 4 freeze  True                                                                      
                  split 1 frozen pipeline LR  True                                                                      
                 split 1 frozen pipeline XGB  True                                                                      
                split 1 frozen pipeline LGBM  True                                                                      
                 split 1 frozen pipeline MLP  True                                                                      
                  split 1 frozen pipeline RF  True                                                                      
                 split 1 frozen pipeline CAT  True                                                                      
                Stage 5 test_pre

In [4]:
# Cell 4 - helpers. model_output, tree_chunk, linear_all, Dedup and perm_rows are the Stage 6 functions (S15b, k, l),
# unchanged, so the native setting follows the Stage 6 code path.
def write_atomic(path, blob):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        f.write(blob)
    os.replace(tmp, path)


def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()


def read_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def model_output(m, pipe, X):
    # the output each explainer decomposes (D9: each model's native scale)
    clf = pipe.named_steps["clf"] if hasattr(pipe, "named_steps") else None
    if m in ("DT", "RF"):
        return clf.predict_proba(X)[:, 1]
    if m == "XGB":
        return clf.predict(X, output_margin=True).astype(float)
    if m == "LGBM":
        return clf.predict(X, raw_score=True)
    if m == "CAT":
        return clf.predict(X, prediction_type="RawFormulaVal")
    if m == "LR":
        return pipe.decision_function(X)
    return pipe.predict_proba(X)[:, 1]


def tree_chunk(path, m, Xc, Xbg, k):
    import time, warnings
    import joblib, numpy as np, shap
    t0, c0 = time.perf_counter(), time.process_time()
    pipe = joblib.load(path)
    clf = pipe.named_steps["clf"]
    if m == "RF":
        clf.set_params(n_jobs=1)
    elif m in ("XGB", "LGBM"):
        clf.set_params(n_jobs=1)                            # CatBoost cannot change parameters after fitting
    obj = clf.get_booster() if m == "XGB" else clf          # shap 0.52 + xgboost 3.4.1: the sklearn wrapper fails (S15b)
    import contextlib, io
    with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):   # no progress bars from workers
        warnings.simplefilter("ignore")
        ex = shap.TreeExplainer(obj, shap.maskers.Independent(Xbg, max_samples=len(Xbg)),   # all background rows (S15b)
                                feature_perturbation="interventional", model_output="raw")
        e = ex(Xc, check_additivity=False)
        fx = model_output(m, pipe, Xc)
    v, b = np.asarray(e.values), np.asarray(e.base_values, dtype=float)
    if v.ndim == 3:                                         # DT / RF: per-class output, keep the positive class (S6)
        v, b = v[:, :, 1], b[:, 1]
    b = np.broadcast_to(b, (len(Xc),)).astype(float)
    return k, v.astype(float), b, np.asarray(fx, float), time.process_time() - c0, time.perf_counter() - t0


def linear_all(path, Xc, Xbg):
    t0 = time.perf_counter()
    pipe = joblib.load(path)
    prep, clf = pipe.named_steps["prep"], pipe.named_steps["clf"]
    names = list(prep.get_feature_names_out())               # the scaled columns come first (S15b)
    Zbg, Z = prep.transform(Xbg), prep.transform(Xc)
    e = shap.LinearExplainer(clf, shap.maskers.Independent(Zbg, max_samples=len(Zbg)))(Z)
    order = [names.index(f) for f in FEATURES]
    v = np.asarray(e.values)[:, order]
    b = np.broadcast_to(np.asarray(e.base_values, float), (len(Xc),)).astype(float)
    return v, b, model_output("LR", pipe, Xc), time.perf_counter() - t0


class Dedup:
    # exact cache of repeated rows within one call (the model is deterministic, so values are unchanged)
    def __init__(self, pipe):
        self.pipe = pipe

    def __call__(self, Z):
        U, inv = np.unique(Z, axis=0, return_inverse=True)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=DeprecationWarning)
            p = self.pipe.predict_proba(pd.DataFrame(U, columns=FEATURES))[:, 1]
        return p[inv.ravel()]


class DedupMargin(Dedup):
    # XGB check only (split 1): the same wrapper on XGBoost's log-odds output
    def __call__(self, Z):
        U, inv = np.unique(Z, axis=0, return_inverse=True)
        p = self.pipe.named_steps["clf"].predict(pd.DataFrame(U, columns=FEATURES), output_margin=True).astype(float)
        return p[inv.ravel()]


def perm_rows(ex, f, Xc, ids, seed):
    vals, base = [], []
    for x, r_ in zip(Xc, ids):
        np.random.seed(row_seed(seed, r_))                  # shap draws permutations from numpy's global generator
        e = ex(x[None, :], max_evals=MAX_EVALS, silent=True)
        vals.append(e.values[0]); base.append(float(np.asarray(e.base_values).ravel()[0]))
    return np.array(vals, float), np.array(base), f(Xc)


def load_pipe(path, m):
    pipe = joblib.load(path)
    if m == "RF":
        pipe.set_params(clf__n_jobs=1)                      # S12e: single-threaded RF prediction
    return pipe


def predict_p(pipe, X):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        return pipe.predict_proba(X)[:, 1]


def shap_frame(rows, V, Bv, FX):
    df = pd.DataFrame(V, columns=FEATURES)
    df.insert(0, "additivity_error", np.abs(Bv + V.sum(axis=1) - FX))
    df.insert(0, "model_output", FX)
    df.insert(0, "base_value", Bv)
    df.insert(0, "row_id", rows)
    return df


def csv_bytes(df):
    return df.to_csv(index=False, float_format="%.17g", lineterminator="\n").encode("utf-8")

In [5]:
# Cell 5 - index sets per split (split 1: Stage 1 files; splits 2-5: the R4 index files)
IDX = {}
for k in SPLITS:
    if k == 1:
        d = pd.read_csv(os.path.join(IDX1, "dev_folds.csv"))
        dev = d.row_id.to_numpy()
        bg = pd.read_csv(os.path.join(IDX1, "shap_background_500_dev.csv")).row_id.to_numpy()
        bg100 = pd.read_csv(os.path.join(IDX1, "shap_perm_background_100_dev.csv")).row_id.to_numpy()
        core = pd.read_csv(os.path.join(IDX1, "shap_perm_core_500_test.csv")).row_id.to_numpy()
        test = PRED.row_id.to_numpy()
    else:
        p = os.path.join(R4, "indices", f"split{k}")
        dev = pd.read_csv(os.path.join(p, "dev_folds.csv")).row_id.to_numpy()
        test = pd.read_csv(os.path.join(p, "test_rows.csv")).row_id.to_numpy()
        bg = pd.read_csv(os.path.join(p, "shap_background_500_dev.csv")).row_id.to_numpy()
        bg100 = pd.read_csv(os.path.join(p, "shap_perm_background_100_dev.csv")).row_id.to_numpy()
        core = pd.read_csv(os.path.join(p, "test_core_500.csv")).row_id.to_numpy()
    assert len(core) == 500 and int(y_all[core].sum()) == 70 and len(bg) == 500 and len(bg100) == 100
    assert set(bg100) <= set(bg) <= set(dev) and set(core) <= set(test) and not set(dev) & set(test)
    IDX[k] = {"dev": np.sort(dev), "test": np.sort(test), "bg": np.sort(bg), "bg100": np.sort(bg100), "core": np.sort(core)}
    print(f"split {k}: dev {len(dev)}, test {len(test)}, background 500/100, core 500 (70 positive)")

split 1: dev 202944, test 50736, background 500/100, core 500 (70 positive)
split 2: dev 202944, test 50736, background 500/100, core 500 (70 positive)
split 3: dev 202944, test 50736, background 500/100, core 500 (70 positive)
split 4: dev 202944, test 50736, background 500/100, core 500 (70 positive)
split 5: dev 202944, test 50736, background 500/100, core 500 (70 positive)


In [6]:
# Cell 6 - models (C2): load or refit, then exact reproduction of the stored test predictions (full test set)
def r4_summary(k, m):
    return read_json(os.path.join(R4, f"split{k}", m, "summary.json"))


def r4_test_p(k, m):
    return pd.read_csv(os.path.join(R4, f"split{k}", m, "test_predictions.csv"), float_precision="round_trip")


def configured(m, cid):
    p = tc.make_pipeline(m, n_jobs=CAT_THREADS if m == "CAT" else -1)
    p.set_params(**tc.sampled_configs(m)[cid])
    return p


def save_model(obj, name):
    path = os.path.join(MODELS_DIR, name)
    tmp = path + ".tmp"
    joblib.dump(obj, tmp); os.replace(tmp, path)
    return path


PIPE = {}                                      # (split, model) -> path of the model explained in R5
REPRO_ROWS = []
for k in SPLITS:
    rec_path = os.path.join(OUT, f"split{k}", "reproduction", "done.json")
    if os.path.exists(rec_path):               # resume: the models were checked in an earlier session
        rec = read_json(rec_path)
        for r in rec["models"]:
            path = os.path.join(PROJECT_DIR, r["path"]) if not os.path.isabs(r["path"]) else r["path"]
            if sha(path) != r["model_sha256"]:
                raise SystemExit(f"split {k} {r['model']}: model file changed since its reproduction check; delete {rec_path}")
            PIPE[(k, r["model"])] = path
            REPRO_ROWS.append(r)
        print(f"split {k}: models verified (checked earlier)")
        continue
    I, rows = IDX[k], []
    Xt = X_all.iloc[I["test"]]
    for m in MODELS6:
        t0 = time.perf_counter()
        if k == 1:
            path, source, want = PIPE1[m], "frozen thesis pipeline", PRED[f"p_{m}"].to_numpy()
        else:
            sm, tp = r4_summary(k, m), r4_test_p(k, m)
            assert np.array_equal(tp.row_id.to_numpy(), I["test"])
            want = tp.p.to_numpy()
            r4p = os.path.join(R4, "pipelines", f"split{k}_{m}.joblib")
            if MODEL_SOURCE == "r4_pipelines" and os.path.exists(r4p):
                if sha(r4p) != sm["pipeline_sha256"]:
                    raise SystemExit(f"split {k} {m}: {r4p} differs from summary.json pipeline_sha256")
                path, source = r4p, "R4 pipeline"
                r4_note = "hash = summary.json"
            else:
                pipe = configured(m, sm["config_id"]).fit(X_all.iloc[I["dev"]], y_all[I["dev"]])
                path, source = save_model(pipe, f"split{k}_{m}.joblib"), f"refit (config {sm['config_id']})"
            r4_note = ("absent" if not os.path.exists(r4p) else "hash = summary.json" if sha(r4p) == sm["pipeline_sha256"] else "hash differs from summary.json")
        p = predict_p(load_pipe(path, m), Xt)
        d = np.abs(p - want)
        tol = REFIT_TOL if (TEST_MODE and k in (1, 2)) else 0.0             # test: splits 1-2 hold PC predictions (last-bit differences, P2-002)
        ok = d.max() <= tol or (TEST_MODE and m == "XGB" and k in (1, 2))   # test: PC XGB cannot be reproduced here (P1-008, P2-003)
        rows.append({"split": k, "model": m, "source": source, "r4_pipeline_on_pc": "-" if k == 1 else r4_note, "path": os.path.relpath(path, PROJECT_DIR),
                     "model_sha256": sha(path), "max_abs_diff": float(d.max()), "rows_differing": int((d > 0).sum()),
                     "exact": bool(d.max() == 0), "pass": bool(ok), "seconds": round(time.perf_counter() - t0, 1)})
        print(rows[-1], flush=True)
    # ALT_XGB: refit (R4 saved no baseline models), wrapped in a passthrough pipeline so the Stage 6 code path applies
    t0 = time.perf_counter()
    bp = pd.read_csv(os.path.join(R4, f"split{k}", "baselines", "baselines_test_predictions.csv"), float_precision="round_trip")
    assert np.array_equal(bp.row_id.to_numpy(), I["test"])
    est = XGBClassifier(**ALT_XGB_PARAMS).fit(X_all.iloc[I["dev"]].to_numpy(), y_all[I["dev"]])
    p = est.predict_proba(Xt.to_numpy())[:, 1]
    d = np.abs(p - bp.p_ALT_XGB.to_numpy())
    alt = Pipeline([("prep", "passthrough"), ("clf", est)])
    path = save_model(alt, f"split{k}_ALT_XGB.joblib")
    ok = d.max() == 0 or (TEST_MODE and k == 2)                             # test: the real PC ALT_XGB (split 2) differs here
    rows.append({"split": k, "model": "ALT_XGB", "source": "refit (Altamimi settings)", "r4_pipeline_on_pc": "not saved by R4", "path": os.path.relpath(path, PROJECT_DIR),
                 "model_sha256": sha(path), "max_abs_diff": float(d.max()), "rows_differing": int((d > 0).sum()),
                 "exact": bool(d.max() == 0), "pass": bool(ok), "seconds": round(time.perf_counter() - t0, 1)})
    print(rows[-1], flush=True)
    R = pd.DataFrame(rows)
    write_atomic(os.path.join(OUT, f"split{k}", "reproduction", "reproduction.csv"),
                 R.to_csv(index=False, lineterminator="\n").encode("utf-8"))
    if not R["pass"].all():
        raise SystemExit(f"split {k}: models do not reproduce the stored test predictions {list(R[~R['pass']].model)}; no SHAP is run.")
    write_atomic(rec_path, json.dumps({"models": rows, "machine": MACHINE,
                                       "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, indent=1).encode("utf-8"))
    for r in rows:
        PIPE[(k, r["model"])] = os.path.join(PROJECT_DIR, r["path"])
    REPRO_ROWS += rows
print(pd.DataFrame(REPRO_ROWS)[["split", "model", "source", "max_abs_diff", "exact", "pass"]].to_string(index=False))

{'split': 1, 'model': 'LR', 'source': 'frozen thesis pipeline', 'r4_pipeline_on_pc': '-', 'path': 'stage3_outputs\\models\\LR\\pipeline_LR.joblib', 'model_sha256': '2deb4902b2c4030c3d9a5cfcdc8474d9a8754462fd6378e65cc6711ca1c90966', 'max_abs_diff': 0.0, 'rows_differing': 0, 'exact': True, 'pass': True, 'seconds': 0.0}
{'split': 1, 'model': 'XGB', 'source': 'frozen thesis pipeline', 'r4_pipeline_on_pc': '-', 'path': 'stage3_outputs\\models\\XGB\\pipeline_XGB.joblib', 'model_sha256': 'f397dc6db4095323b18c007b9cd8f22668c6b89c2bce53ae760a9460bdc7b062', 'max_abs_diff': 0.0, 'rows_differing': 0, 'exact': True, 'pass': True, 'seconds': 0.2}
{'split': 1, 'model': 'LGBM', 'source': 'frozen thesis pipeline', 'r4_pipeline_on_pc': '-', 'path': 'stage3_outputs\\models\\LGBM\\pipeline_LGBM.joblib', 'model_sha256': '4b5d7646db5457245c4c8f8a7f6a360cfb2d8f8de50176e92aeffc18bf671997', 'max_abs_diff': 0.0, 'rows_differing': 0, 'exact': True, 'pass': True, 'seconds': 0.4}
{'split': 1, 'model': 'MLP', 'sour

In [7]:
# Cell 7 - regression check (C3): the R5 code on the first 20 split-1 core rows vs the Stage 6 values
REG_PATH = os.path.join(OUT, "split1", "regression_check.csv")
if 1 in SPLITS and not os.path.exists(REG_PATH):
    ids = IDX[1]["core"][:REGRESSION_ROWS]
    Xc, XBG1, XBG1_100 = X_all.iloc[ids].reset_index(drop=True), X_all.iloc[IDX[1]["bg"]].reset_index(drop=True), X_all.iloc[IDX[1]["bg100"]].reset_index(drop=True)
    reg = []
    for m in MODELS6:
        ref = pd.read_csv(S6_FILES[m], float_precision="round_trip").set_index("row_id").loc[ids]
        if m == "LR":
            v, b, fx, _ = linear_all(PIPE1[m], Xc, XBG1)
        elif m in TREE:
            _, v, b, fx, _, _ = tree_chunk(PIPE1[m], m, Xc, XBG1, 0)
        else:
            pipe = load_pipe(PIPE1[m], m); f = Dedup(pipe)
            ex = shap.PermutationExplainer(f, shap.maskers.Independent(XBG1_100.to_numpy(), max_samples=len(XBG1_100)))
            v, b, fx = perm_rows(ex, f, Xc.to_numpy(), ids, STAGE6_PERM_SEED)
        dv = float(np.abs(v - ref[FEATURES].to_numpy()).max())
        db = float(np.abs(b - ref.base_value.to_numpy()).max())
        dfx = float(np.abs(fx - ref.model_output.to_numpy()).max())
        # permutation: SHAP and base values must be identical; the model output column may differ in the last bit,
        # because Stage 6 predicted it on another row batch (MLP batch size); tree/linear within 1e-10
        tol = 0.0 if m == "MLP" else 1e-10
        ok = (dv <= tol and db <= tol and dfx <= 1e-12) if m == "MLP" else max(dv, db, dfx) <= tol
        reg.append({"model": m, "rows": len(ids), "max_abs_diff_values": dv, "max_abs_diff_base": db, "max_abs_diff_output": dfx,
                    "tolerance": tol, "pass": ok})
        print(reg[-1], flush=True)
    REG = pd.DataFrame(reg)
    if not REG["pass"].all():
        raise SystemExit("Regression check failed: R5 does not reproduce Stage 6 " + str(list(REG[~REG["pass"]].model)))
    write_atomic(REG_PATH, REG.to_csv(index=False, float_format="%.3g", lineterminator="\n").encode("utf-8"))
if os.path.exists(REG_PATH):
    print(pd.read_csv(REG_PATH).to_string(index=False))

# split-1 native values: the Stage 6 rows of the 500-row core, unchanged (17 digits)
if 1 in SPLITS:
    for m in MODELS6:
        rel = os.path.join("split1", "native", f"SHAP_{m}.csv")
        dpath = os.path.join(OUT, "split1", "native", f"SHAP_{m}.done.json")
        if os.path.exists(dpath) and sha(os.path.join(OUT, rel)) == read_json(dpath)["sha256"]:
            continue
        ref = pd.read_csv(S6_FILES[m], float_precision="round_trip")
        sub = ref[ref.row_id.isin(IDX[1]["core"])].sort_values("row_id")
        if not TEST_MODE and len(sub) != 500:
            raise SystemExit(f"Stage 6 SHAP_{m}.csv does not cover the 500-row core")
        sub = shap_frame(sub.row_id.to_numpy(), sub[FEATURES].to_numpy(), sub.base_value.to_numpy(), sub.model_output.to_numpy())
        blob = csv_bytes(sub)
        write_atomic(os.path.join(OUT, rel), blob)
        write_atomic(dpath, json.dumps({"file": rel.replace(os.sep, "/"), "sha256": sha_bytes(blob), "rows": len(sub), "source": "Stage 6",
                                        "stage6_sha256": sha(S6_FILES[m]), "output_scale": SCALE_NATIVE[m],
                                        "max_additivity_error": float(sub.additivity_error.max()), "machine": MACHINE}, indent=1).encode("utf-8"))
    print("split 1 native: Stage 6 core rows written for", MODELS6)

{'model': 'LR', 'rows': 20, 'max_abs_diff_values': 0.0, 'max_abs_diff_base': 0.0, 'max_abs_diff_output': 0.0, 'tolerance': 1e-10, 'pass': True}
{'model': 'XGB', 'rows': 20, 'max_abs_diff_values': 0.0, 'max_abs_diff_base': 0.0, 'max_abs_diff_output': 0.0, 'tolerance': 1e-10, 'pass': True}
{'model': 'LGBM', 'rows': 20, 'max_abs_diff_values': 0.0, 'max_abs_diff_base': 0.0, 'max_abs_diff_output': 0.0, 'tolerance': 1e-10, 'pass': True}
{'model': 'MLP', 'rows': 20, 'max_abs_diff_values': 0.0, 'max_abs_diff_base': 0.0, 'max_abs_diff_output': 0.0, 'tolerance': 0.0, 'pass': True}
{'model': 'RF', 'rows': 20, 'max_abs_diff_values': 0.0, 'max_abs_diff_base': 0.0, 'max_abs_diff_output': 0.0, 'tolerance': 1e-10, 'pass': True}
{'model': 'CAT', 'rows': 20, 'max_abs_diff_values': 0.0, 'max_abs_diff_base': 0.0, 'max_abs_diff_output': 0.0, 'tolerance': 1e-10, 'pass': True}
model  rows  max_abs_diff_values  max_abs_diff_base  max_abs_diff_output    tolerance  pass
   LR    20                    0         

In [8]:
# Cell 8 - task list (native splits 2-5, ALT_XGB native splits 1-5, common splits 1-5, XGB check split 1)
TASKS = []
for k in SPLITS:
    if k != 1:
        TASKS += [("native", k, m) for m in MODELS6]
    TASKS.append(("native", k, "ALT_XGB"))
    TASKS += [("common", k, m) for m in MODELS6]
    if k == 1:
        TASKS += [("xgbcheck", 1, "XGB_tree100"), ("xgbcheck", 1, "XGB_perm_logodds")]


def explainer_kind(t):
    s, k, m = t
    if s == "native":
        return "linear" if m == "LR" else "perm" if m == "MLP" else "tree"
    if s == "xgbcheck":
        return "tree" if m == "XGB_tree100" else "perm"
    return "perm"


def task_name(t):
    return f"split{t[1]}_{t[0]}_{t[2]}"


def task_rows(t):
    rows = IDX[t[1]]["core"]
    lim = MAX_ROWS_MODEL.get(t[2].replace("XGB_tree100", "XGB").replace("XGB_perm_logodds", "XGB"), MAX_ROWS)
    return rows[:lim] if lim else rows


def task_model(t):
    s, k, m = t
    if s == "xgbcheck":
        return "XGB", PIPE[(1, "XGB")]
    return ("XGB" if m == "ALT_XGB" else m), PIPE[(k, m)]


def task_bg(t):
    s, k, m = t
    key = "bg" if (s == "native" and m != "MLP") else "bg100"
    return key, IDX[k][key]


def task_seed(t):
    return perm_seed(t[1]) if explainer_kind(t) == "perm" else None


def task_files(t):
    rel = os.path.join(f"split{t[1]}", t[0], f"SHAP_{t[2]}.csv")
    return rel, os.path.join(OUT, f"split{t[1]}", t[0], f"SHAP_{t[2]}.done.json")


def is_done(t):
    rel, dp = task_files(t)
    if not os.path.exists(dp):
        return False
    if sha(os.path.join(OUT, rel)) != read_json(dp)["sha256"]:
        raise SystemExit(f"{rel} differs from its .done.json; delete both and rerun this task.")
    return True


# Planning rates for the PC (seconds per explained row; tree rates per row on one core), from the full-size R5 workspace
# test (2026-11-13) scaled by 1.15 (MLP permutation SHAP: 0.045 s per row in the workspace, 0.052 on the PC, S7-030) and,
# for RF, the Stage 6 PC run (S6-065). Used only for the plan and the time estimate; not results.
RATE_TREE = {"XGB": 0.08, "LGBM": 0.13, "CAT": 0.25, "RF": 2.5, "ALT_XGB": 0.035, "XGB_tree100": 0.065}
RATE_PERM = json.loads(os.environ.get("R5_RATE_PERM", "{}")) or {"LR": 0.05, "XGB": 0.10, "LGBM": 0.21, "MLP": 0.052,
                                                                 "RF": 0.35, "CAT": 0.045, "XGB_perm_logodds": 0.10}


def planned_seconds(t):
    n = len(task_rows(t)); kind = explainer_kind(t)
    if kind == "linear":
        return 5
    if kind == "tree":
        return n * RATE_TREE[t[2]] / (0.8 * TREE_WORKERS) + 10
    return n * RATE_PERM[t[2]] + 5


TASKS = sorted(TASKS, key=lambda t: (planned_seconds(t), t[1]))           # cheapest first
PLAN = pd.DataFrame([{"task": task_name(t), "explainer": explainer_kind(t), "rows": len(task_rows(t)),
                      "planned_min": planned_seconds(t) / 60, "status": "done" if is_done(t) else "to run"} for t in TASKS])
print(PLAN.round(2).to_string(index=False))
todo = PLAN.loc[PLAN.status == "to run", "planned_min"].sum()
print(f"\nPlanned time for the remaining tasks: {todo:.0f} min (about {todo / 60:.1f} h); allow up to about 1.5x.")

                            task explainer  rows  planned_min status
                split2_native_LR    linear   500         0.08 to run
                split3_native_LR    linear   500         0.08 to run
                split4_native_LR    linear   500         0.08 to run
                split5_native_LR    linear   500         0.08 to run
           split1_native_ALT_XGB      tree   500         0.23 to run
           split2_native_ALT_XGB      tree   500         0.23 to run
           split3_native_ALT_XGB      tree   500         0.23 to run
           split4_native_ALT_XGB      tree   500         0.23 to run
           split5_native_ALT_XGB      tree   500         0.23 to run
     split1_xgbcheck_XGB_tree100      tree   500         0.28 to run
               split2_native_XGB      tree   500         0.31 to run
               split3_native_XGB      tree   500         0.31 to run
               split4_native_XGB      tree   500         0.31 to run
               split5_native_XGB  

In [9]:
# Cell 9 - run every task (resumable). Progress and the estimated time remaining are printed as chunks finish.
SPEED, LAST = [], [0.0]


def progress(msg, final=False):
    now = time.perf_counter()
    if final or now - LAST[0] >= 30:
        print(msg, flush=True); LAST[0] = now


def eta_text(cur_left, after):
    f = float(np.median(SPEED)) if SPEED else 1.0
    rest = sum(planned_seconds(t) for t in after if not is_done(t)) * f
    return f"task ETA {cur_left / 60:.1f} min; all remaining about {(cur_left + rest) / 60:.0f} min"


def after_chunk(name, k):                     # test hook: simulated interruption
    if STOP_AFTER and STOP_AFTER in (f"{name}:{k}", f"{name}:any"):
        raise SystemExit(f"TEST INTERRUPT after {name} chunk {k}")


def manifest(t, rows, chunk, bg_key, bg):
    wd = os.path.join(WORK, task_name(t))
    os.makedirs(wd, exist_ok=True)
    mm, path = task_model(t)
    man = {"rows_sha256": sha_bytes(np.asarray(rows, np.int64).tobytes()), "n_rows": len(rows), "chunk": chunk,
           "model_sha256": sha(path), "background": bg_key, "background_sha256": sha_bytes(np.asarray(bg, np.int64).tobytes()),
           "seed": task_seed(t), "max_evals": MAX_EVALS, "shap": shap.__version__}
    mp = os.path.join(wd, "manifest.json")
    if os.path.exists(mp):
        if read_json(mp) != man:
            raise SystemExit(f"{mp} was written with other settings. Delete {wd} and rerun.")
    else:
        write_atomic(mp, json.dumps(man, indent=1).encode("utf-8"))
    return wd


def load_chunks(wd, n):
    got = {}
    for k in range(n):
        p = os.path.join(wd, f"c{k:04d}.npz")
        if os.path.exists(p):
            with np.load(p) as z:
                got[k] = {key: z[key] for key in z.files}
    return got


def save_chunk(wd, k, **arrs):
    p = os.path.join(wd, f"c{k:04d}.npz")
    tmp = p + ".tmp.npz"
    np.savez(tmp, **arrs)
    os.replace(tmp, p)


def run_task(t, after):
    s, k, m = t
    kind, (mm, path), (bg_key, bg) = explainer_kind(t), task_model(t), task_bg(t)
    rows = task_rows(t)
    chunk = 10 ** 9 if kind == "linear" else TREE_CHUNK if kind == "tree" else PERM_CHUNK
    wd = manifest(t, rows, chunk, bg_key, bg)
    Xr = X_all.iloc[rows].reset_index(drop=True)
    Xbg = X_all.iloc[bg].reset_index(drop=True)
    starts = list(range(0, len(rows), chunk))
    got = load_chunks(wd, len(starts))
    todo = [c for c in range(len(starts)) if c not in got]
    t0, rows_done, done_here = time.perf_counter(), sum(len(got[c]["v"]) for c in got), 0
    n_left = sum(len(rows[starts[c]:starts[c] + chunk]) for c in todo)
    print(f"[{task_name(t)}] {len(rows)} rows, {len(starts)} chunks, {len(got)} already saved", flush=True)
    if kind == "linear":
        if todo:
            v, b, fx, w = linear_all(path, Xr, Xbg)
            save_chunk(wd, 0, v=v, b=b, fx=fx, wall=np.array([w]), cpu=np.array([w]))
    elif kind == "tree":
        jobs = (delayed(tree_chunk)(path, mm, Xr.iloc[starts[c]:starts[c] + chunk], Xbg, c) for c in todo)
        for c, v, b, fx, cpu, w in Parallel(n_jobs=TREE_WORKERS, return_as="generator_unordered")(jobs):
            save_chunk(wd, c, v=v, b=b, fx=fx, wall=np.array([w]), cpu=np.array([cpu]))
            rows_done += len(v); n_left -= len(v); done_here += len(v)
            el = time.perf_counter() - t0
            progress(f"  {rows_done}/{len(rows)} rows, {el / 60:.1f} min; " + eta_text(n_left * el / done_here, after), n_left == 0)
            after_chunk(task_name(t), c)
    else:
        pipe = load_pipe(path, mm)
        f = DedupMargin(pipe) if m == "XGB_perm_logodds" else Dedup(pipe)
        ex = shap.PermutationExplainer(f, shap.maskers.Independent(Xbg.to_numpy(), max_samples=len(Xbg)))
        Xn = Xr.to_numpy()
        for c in todo:
            sl = slice(starts[c], starts[c] + chunk)
            c0, w0 = time.process_time(), time.perf_counter()
            v, b, fx = perm_rows(ex, f, Xn[sl], rows[sl], task_seed(t))
            save_chunk(wd, c, v=v, b=b, fx=fx, wall=np.array([time.perf_counter() - w0]), cpu=np.array([time.process_time() - c0]))
            rows_done += len(v); n_left -= len(v); done_here += len(v)
            el = time.perf_counter() - t0
            progress(f"  {rows_done}/{len(rows)} rows, {el / 60:.1f} min; " + eta_text(n_left * el / done_here, after), n_left == 0)
            after_chunk(task_name(t), c)
    got = load_chunks(wd, len(starts))
    V = np.vstack([got[c]["v"] for c in range(len(starts))])
    Bv = np.concatenate([got[c]["b"] for c in range(len(starts))])
    FX = np.concatenate([got[c]["fx"] for c in range(len(starts))])
    cpu = float(sum(got[c]["cpu"][0] for c in got))
    if kind != "tree":
        wall = float(sum(got[c]["wall"][0] for c in got))
    elif todo == list(range(len(starts))):
        wall = time.perf_counter() - t0
    else:
        wall = float(sum(got[c]["wall"][0] for c in got)) / TREE_WORKERS
    df = shap_frame(rows, V, Bv, FX)
    blob = csv_bytes(df)
    rel, dp = task_files(t)
    write_atomic(os.path.join(OUT, rel), blob)
    scale = "probability" if s == "common" else "log-odds" if s == "xgbcheck" else SCALE_NATIVE[m]
    expl = {"linear": "LinearExplainer (interventional, 500 bg, standardized inputs)",
            "tree": f"TreeExplainer (interventional, {len(bg)} bg)",
            "perm": f"PermutationExplainer ({len(bg)} bg, max_evals {MAX_EVALS})"}[kind]
    rec = {"file": rel.replace(os.sep, "/"), "sha256": sha_bytes(blob), "split": k, "setting": s, "model": m, "rows": len(rows),
           "explainer": expl, "output_scale": scale, "background": bg_key, "seed": task_seed(t), "model_sha256": sha(path),
           "chunks": len(starts), "workers": TREE_WORKERS if kind == "tree" else 1, "wall_s": wall, "cpu_s": cpu,
           "max_additivity_error": float(df.additivity_error.max()), "mean_additivity_error": float(df.additivity_error.mean()),
           "machine": MACHINE, "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    write_atomic(dp, json.dumps(rec, indent=1).encode("utf-8"))
    return rec


t_session = time.perf_counter()
for i, t in enumerate(TASKS):
    if is_done(t):
        continue
    t1 = time.perf_counter()
    rec = run_task(t, TASKS[i + 1:])
    used = time.perf_counter() - t1
    SPEED.append(used / planned_seconds(t))
    print(f"[{task_name(t)}] done in {used / 60:.2f} min; max additivity error {rec['max_additivity_error']:.2e}", flush=True)
print(f"All {len(TASKS)} tasks finished. This session: {(time.perf_counter() - t_session) / 60:.1f} min.")

[split2_native_LR] 500 rows, 1 chunks, 0 already saved
[split2_native_LR] done in 0.00 min; max additivity error 1.78e-15
[split3_native_LR] 500 rows, 1 chunks, 0 already saved
[split3_native_LR] done in 0.00 min; max additivity error 1.33e-15
[split4_native_LR] 500 rows, 1 chunks, 0 already saved
[split4_native_LR] done in 0.00 min; max additivity error 2.22e-15
[split5_native_LR] 500 rows, 1 chunks, 0 already saved
[split5_native_LR] done in 0.00 min; max additivity error 8.88e-16
[split1_native_ALT_XGB] 500 rows, 10 chunks, 0 already saved
  50/500 rows, 0.1 min; task ETA 1.1 min; all remaining about 2 min
  500/500 rows, 0.2 min; task ETA 0.0 min; all remaining about 1 min
[split1_native_ALT_XGB] done in 0.15 min; max additivity error 3.90e-06
[split2_native_ALT_XGB] 500 rows, 10 chunks, 0 already saved
  500/500 rows, 0.1 min; task ETA 0.0 min; all remaining about 1 min
[split2_native_ALT_XGB] done in 0.06 min; max additivity error 5.61e-06
[split3_native_ALT_XGB] 500 rows, 10 chu

In [10]:
# Cell 10 - additivity table, ALT_XGB top-5 sets, timing, environment log, report
rows, top5, tim = [], [], []
for k in SPLITS:
    for s in ["native", "common", "xgbcheck"]:
        d = os.path.join(OUT, f"split{k}", s)
        if not os.path.isdir(d):
            continue
        for fn in sorted(x for x in os.listdir(d) if x.endswith(".done.json")):
            rec = read_json(os.path.join(d, fn))
            df = pd.read_csv(os.path.join(OUT, rec["file"]), float_precision="round_trip")
            m = fn[len("SHAP_"):-len(".done.json")]
            rows.append({"split": k, "setting": s, "model": m, "explainer": rec.get("explainer", "Stage 6 (" + rec.get("source", "") + ")"),
                         "output_scale": rec["output_scale"], "rows": len(df), "max_abs_additivity_error": df.additivity_error.max(),
                         "mean_abs_additivity_error": df.additivity_error.mean()})
            if "wall_s" in rec:
                tim.append({"task": f"split{k}_{s}_{m}", "rows": len(df), "workers": rec["workers"], "wall_s": rec["wall_s"],
                            "cpu_s": rec["cpu_s"], "wall_s_per_row": rec["wall_s"] / len(df), "machine": rec["machine"]})
            if m == "ALT_XGB":
                imp = df[FEATURES].abs().mean().to_numpy(); nrm = imp / imp.sum()
                order = np.argsort(-nrm, kind="stable")[:5]
                top5.append({"split": k, "rows": len(df), **{f"top{j + 1}": FEATURES[i] for j, i in enumerate(order)},
                             **{f"share{j + 1}": nrm[i] for j, i in enumerate(order)}})
ADD, TOP5, TIM = pd.DataFrame(rows), pd.DataFrame(top5), pd.DataFrame(tim)
write_atomic(os.path.join(OUT, "additivity.csv"), ADD.to_csv(index=False, float_format="%.6g", lineterminator="\n").encode("utf-8"))
write_atomic(os.path.join(OUT, "alt_xgb_top5.csv"), TOP5.to_csv(index=False, float_format="%.6g", lineterminator="\n").encode("utf-8"))
write_atomic(os.path.join(OUT, "timing_r5.csv"), TIM.to_csv(index=False, float_format="%.6g", lineterminator="\n").encode("utf-8"))
write_atomic(os.path.join(OUT, "gates.csv"), GATE_DF.to_csv(index=False, lineterminator="\n").encode("utf-8"))
write_atomic(os.path.join(OUT, "reproduction_all.csv"), pd.DataFrame(REPRO_ROWS).to_csv(index=False, lineterminator="\n").encode("utf-8"))
expected = len(TASKS) + (6 if 1 in SPLITS else 0)
if len(ADD) != expected:
    raise SystemExit(f"{len(ADD)} SHAP files, expected {expected}")
from importlib import metadata
env = {"machine": MACHINE, "python": platform.python_version(), "platform": platform.platform(), "cpu_count": os.cpu_count(),
       "packages": {p: metadata.version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "lightgbm", "catboost", "shap"]},
       "pins_differ": {k: list(v) for k, v in tc.check_pins(PROJECT_DIR).items()}, "test_mode": TEST_MODE, "model_source": MODEL_SOURCE,
       "tree_workers": TREE_WORKERS, "max_rows": MAX_ROWS, "max_rows_model": MAX_ROWS_MODEL, "paper_outer": PAPER_OUTER,
       "permutation_seeds": {k: perm_seed(k) for k in SPLITS}, "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
write_atomic(os.path.join(OUT, f"env_log_r5_{MACHINE}.json"), json.dumps(env, indent=1).encode("utf-8"))
pd.set_option("display.width", 200)
print(ADD.to_string(index=False))
print("\nALT_XGB top-5 sets:\n" + TOP5.to_string(index=False))
print(f"\nmax additivity error: tree/linear {ADD[~ADD.explainer.str.startswith('Perm')].max_abs_additivity_error.max():.2e}; "
      f"permutation {ADD[ADD.explainer.str.startswith('Perm')].max_abs_additivity_error.max():.2e}")
print("\nNext: run  python pack_r5_outputs.py  and upload r5_upload/ (every zip and SHA256SUMS_r5.txt).")

 split  setting            model                                                     explainer output_scale  rows  max_abs_additivity_error  mean_abs_additivity_error
     1   native          ALT_XGB                        TreeExplainer (interventional, 500 bg)     log-odds   500              3.899702e-06               6.384745e-07
     1   native              CAT                                             Stage 6 (Stage 6)     log-odds   500              9.152559e-08               2.158369e-08
     1   native             LGBM                                             Stage 6 (Stage 6)     log-odds   500              1.174312e-07               4.327112e-08
     1   native               LR                                             Stage 6 (Stage 6)     log-odds   500              2.220446e-15               2.304823e-16
     1   native              MLP                                             Stage 6 (Stage 6)  probability   500              1.332268e-15               1.660134e-1